# Lab 8 - Segmentation

**Course**: Deep Learning for Image Analysis

**Class**: M2 IASD App  

**Professor**: Mehyar MLAWEH

**Deadline:** 🗓️ **Saturday, June 6th (23:59)**

------
**Objective:** In this lab you will go from raw images to pixel-precise segmentation masks.  
You will explore classical methods, use the Segment Anything Model (SAM ViT-B) with different prompt strategies, implement evaluation metrics from scratch, and interpret your results.

**Dataset:** Oxford-IIIT Pets (5-breed subset, 50 images)  
**Model:** SAM ViT-B (~375 MB) — the lightweight version of Meta's SAM

---
>⚠️ This notebook is **intentionally incomplete**.  
Whenever you see **`# TODO`**, you are expected to write code.


---
## Part 0 — Installation and Setup
Run the cells below once. They install all required libraries and download the SAM ViT-B checkpoint.


In [ ]:
# Install required libraries
!pip install git+https://github.com/facebookresearch/segment-anything.git -q
!pip install opencv-python-headless matplotlib numpy torchvision tqdm -q
print("✅ Libraries installed")


In [ ]:
import os

# Download SAM ViT-B checkpoint (~375 MB)
checkpoint_path = "sam_vit_b_01ec64.pth"
if not os.path.exists(checkpoint_path):
    print("Downloading SAM ViT-B checkpoint (~375 MB) ...")
    !wget -q --show-progress https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
    print("✅ Checkpoint downloaded")
else:
    print("✅ Checkpoint already exists")


In [ ]:
# Download Oxford-IIIT Pets dataset (images + annotations)
import os

if not os.path.exists("oxford-iiit-pet"):
    os.makedirs("oxford-iiit-pet", exist_ok=True)
    print("Downloading Oxford-IIIT Pets images ...")
    !wget -q --show-progress https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz
    !wget -q --show-progress https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz
    !tar -xzf images.tar.gz -C oxford-iiit-pet/
    !tar -xzf annotations.tar.gz -C oxford-iiit-pet/
    print("✅ Dataset downloaded and extracted")
else:
    print("✅ Dataset already exists")


In [ ]:
# Global imports — run this cell before anything else
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import torch
import os
import random
from pathlib import Path
from PIL import Image

# SAM imports
from segment_anything import sam_model_registry, SamPredictor, SamAutomaticMaskGenerator

# Reproducibility
random.seed(42)
np.random.seed(42)

print("✅ All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")


---
## Part 1 — Data Exploration

Before building any model, we need to understand our data.  
The Oxford-IIIT Pets dataset contains images of cats and dogs with **trimap masks**.

A trimap has three pixel values:
- `1` → foreground (the pet)
- `2` → background
- `3` → boundary / uncertain region

We will work with a **5-breed subset** of 50 images for speed.


In [ ]:
# ── Helper: load dataset paths (5 breeds, 10 images each) ──────────────────
BREEDS = ["Abyssinian", "Bengal", "Bombay", "british_shorthair", "german_shorthaired"]
IMG_DIR  = Path("oxford-iiit-pet/images")
MASK_DIR = Path("oxford-iiit-pet/annotations/trimaps")

def load_dataset(breeds=BREEDS, n_per_breed=10):
    """Return list of (image_path, mask_path) tuples for selected breeds."""
    pairs = []
    for breed in breeds:
        imgs = sorted(IMG_DIR.glob(f"{breed}_*.jpg"))[:n_per_breed]
        for img_path in imgs:
            mask_path = MASK_DIR / (img_path.stem + ".png")
            if mask_path.exists():
                pairs.append((img_path, mask_path))
    random.shuffle(pairs)
    return pairs

dataset = load_dataset()
print(f"✅ Dataset loaded: {len(dataset)} image-mask pairs")


In [ ]:
# ── Helper: load image and mask ─────────────────────────────────────────────
def load_image(path):
    """Load image as RGB numpy array."""
    img = cv2.imread(str(path))
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def load_mask(path):
    """Load trimap mask as numpy array (values: 1, 2, 3)."""
    return np.array(Image.open(path))

# Quick sanity check
img, mask = load_image(dataset[0][0]), load_mask(dataset[0][1])
print(f"Image shape : {img.shape}  dtype: {img.dtype}")
print(f"Mask shape  : {mask.shape}  unique values: {np.unique(mask)}")


### TODO 1 — Visualize 5 images with their ground truth masks
Complete the function below to display 5 random image-mask pairs side by side.  
Each row should show: **original image | trimap mask | colored overlay**.

**Hint:** use `plt.imshow()`. For the overlay, you can use a colormap like `'jet'` on the mask.


In [ ]:
def visualize_samples(dataset, n=5):
    """
    Display n samples from the dataset.
    Each row: original image | trimap mask | overlay
    """
    samples = random.sample(dataset, n)
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    fig.suptitle("Dataset Samples: Image | Trimap Mask | Overlay", fontsize=14, fontweight='bold')

    for i, (img_path, mask_path) in enumerate(samples):
        img  = load_image(img_path)
        mask = load_mask(mask_path)

        # Display the original image in axes[i, 0]
        # Display the trimap mask in axes[i, 1] with colormap 'jet'
        # Display a semi-transparent mask overlay on the image in axes[i, 2]
        # Add a title to each column
        # ────────────────────────────────────────────────────────────────────
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f"Original — {img_path.name}")
        axes[i, 0].axis('off')

        axes[i, 1].imshow(mask, cmap='jet')
        axes[i, 1].set_title("Trimap Mask")
        axes[i, 1].axis('off')

        axes[i, 2].imshow(img)
        axes[i, 2].imshow(mask, cmap='jet', alpha=0.5)
        axes[i, 2].set_title("Overlay")
        axes[i, 2].axis('off')

    plt.tight_layout()
    plt.show()

visualize_samples(dataset, n=5)


### TODO 2 — Convert trimap to binary mask
The trimap has values 1 (pet), 2 (background), 3 (boundary).  
Convert it to a **binary mask**: `1` where the pixel is pet, `0` everywhere else.  
Treat boundary pixels (value 3) as foreground.


In [ ]:
def trimap_to_binary(trimap):
    """
    Convert trimap mask to binary foreground mask.
    Args:
        trimap: numpy array with values 1 (pet), 2 (background), 3 (boundary)
    Returns:
        binary mask: numpy array with values 0 and 1
    """
    # Hint: use numpy boolean operations
    return ((trimap == 1) | (trimap == 3)).astype(np.uint8)
    # ────────────────────────────────────────────────────────────────────────

# Test
sample_mask = load_mask(dataset[0][1])
binary = trimap_to_binary(sample_mask)
print(f"Trimap unique values : {np.unique(sample_mask)}")
print(f"Binary unique values : {np.unique(binary)}")
print(f"Foreground pixels    : {binary.sum()} / {binary.size} ({100*binary.mean():.1f}%)")


### TODO 3 — Plot foreground pixel distribution
For each of the 50 images, compute the percentage of foreground pixels.  
Plot a **bar chart** showing this distribution across all images.  
Add a horizontal line showing the mean foreground percentage.


In [ ]:
def plot_foreground_distribution(dataset):
    """
    For each image compute % of foreground pixels and plot as bar chart.
    """
    # 1. Loop over dataset, load each mask, convert to binary, compute fg %
    # 2. Store percentages in a list
    # 3. Add a red horizontal dashed line for the mean
    fg_percentages = []
    for img_path, mask_path in dataset:
        mask = load_mask(mask_path)
        binary = trimap_to_binary(mask)
        fg_percentages.append(100 * binary.mean())

    mean_fg = np.mean(fg_percentages)
    plt.figure(figsize=(14, 4))
    plt.bar(range(len(fg_percentages)), fg_percentages, color='steelblue')
    plt.axhline(mean_fg, color='red', linestyle='--', label=f'Mean: {mean_fg:.1f}%')
    plt.xlabel('Image index')
    plt.ylabel('Foreground %')
    plt.title('Foreground Pixel Distribution across Dataset')
    plt.legend()
    plt.tight_layout()
    plt.show()
    # ────────────────────────────────────────────────────────────────────────

plot_foreground_distribution(dataset)


---
## Part 2 — Classical Segmentation: Why Naive Methods Fail

Before using SAM, we try two classical segmentation approaches:
- **Color thresholding** in HSV colorspace
- **Otsu's automatic thresholding** on grayscale

These methods require no learning, no data
But they have fundamental limitations, you will measure exactly how limited they are.


### TODO 4 — Color thresholding in HSV
Convert the image to HSV and threshold on the **saturation channel** to isolate the pet.  
Pets typically have higher saturation than backgrounds like grass or walls.


In [ ]:
def color_threshold_segment(image_rgb):
    """
    Segment foreground using HSV saturation thresholding.
    Args:
        image_rgb: numpy array (H, W, 3) in RGB
    Returns:
        binary mask: numpy array (H, W) with values 0 and 1
    """
    # 1. Convert RGB image to HSV using cv2.cvtColor
    # 2. Extract the saturation channel (index 1)
    # 3. Apply a threshold: pixels with saturation > 60 are foreground
    # 4. Return binary mask (0 or 1, not 0 or 255)
    hsv = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV)
    saturation = hsv[:, :, 1]
    binary_mask = (saturation > 60).astype(np.uint8)
    return binary_mask
    # ────────────────────────────────────────────────────────────────────────

# Test on one image
test_img = load_image(dataset[0][0])
pred = color_threshold_segment(test_img)
print(f"Predicted mask shape: {pred.shape}, unique: {np.unique(pred)}")


### TODO 5 — Morphological cleaning
Raw thresholding produces noisy masks with holes and isolated pixels.  
Apply **erosion followed by dilation** (opening) to clean the mask.


In [ ]:
def morphological_clean(binary_mask, kernel_size=5, iterations=2):
    """
    Clean a binary mask using morphological opening (erosion then dilation).
    Args:
        binary_mask : numpy array (H, W) with values 0 and 1
        kernel_size : size of the structuring element
        iterations  : number of times to apply each operation
    Returns:
        cleaned binary mask
    """
    # 1. Create a kernel using cv2.getStructuringElement
    # 2. Convert mask to uint8 (multiply by 255)
    # 3. Apply cv2.erode() then cv2.dilate() with the kernel
    # 4. Convert back to binary (0 or 1) and return
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    mask_uint8 = (binary_mask * 255).astype(np.uint8)
    eroded = cv2.erode(mask_uint8, kernel, iterations=iterations)
    dilated = cv2.dilate(eroded, kernel, iterations=iterations)
    return (dilated > 0).astype(np.uint8)
    # ────────────────────────────────────────────────────────────────────────


### TODO 6 — Visualize thresholding results
Show 3 images with: **original | predicted mask | colored overlay on image**.


In [ ]:
def visualize_classical(dataset, segment_fn, clean_fn, title, n=3):
    """
    Visualize classical segmentation results.
    Each row: original image | predicted mask | overlay
    """
    samples = dataset[:n]
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    for i, (img_path, mask_path) in enumerate(samples):
        img    = load_image(img_path)
        gt     = trimap_to_binary(load_mask(mask_path))
        pred   = clean_fn(segment_fn(img))

        # Display: original image | predicted mask | overlay
        # Title each column: "Original", "Predicted Mask", "Overlay"
        # For the overlay: show image with semi-transparent mask on top
        axes[i, 0].imshow(img)
        axes[i, 0].set_title('Original')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(pred, cmap='gray')
        axes[i, 1].set_title('Predicted Mask')
        axes[i, 1].axis('off')

        axes[i, 2].imshow(img)
        axes[i, 2].imshow(pred, cmap='jet', alpha=0.5)
        axes[i, 2].set_title('Overlay')
        axes[i, 2].axis('off')
        # ────────────────────────────────────────────────────────────────────

    plt.tight_layout()
    plt.show()

visualize_classical(dataset, color_threshold_segment, morphological_clean,
                    "Color Thresholding Results", n=3)


### TODO 7 — Otsu's automatic thresholding
Otsu's method automatically finds the optimal threshold on the **grayscale** image.  
It requires no manual parameter tuning.


In [ ]:
def otsu_segment(image_rgb):
    """
    Segment foreground using Otsu's automatic thresholding on grayscale.
    Args:
        image_rgb: numpy array (H, W, 3) in RGB
    Returns:
        binary mask: numpy array (H, W) with values 0 and 1
    """
    # 1. Convert RGB to grayscale using cv2.cvtColor
    # 2. Apply Otsu thresholding: cv2.threshold with cv2.THRESH_OTSU flag
    # 3. Return binary mask (0 or 1)
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return (thresh > 0).astype(np.uint8)
    # ────────────────────────────────────────────────────────────────────────


### TODO 8 — Compare methods side by side
Display 5 images as a grid: **original | color threshold | Otsu | ground truth**  
4 columns, 5 rows.


In [ ]:
def compare_classical_methods(dataset, n=5):
    """
    Compare color thresholding vs Otsu vs ground truth.
    Grid: n rows x 4 columns (original, color thresh, otsu, ground truth)
    """
    samples = dataset[:n]
    fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
    fig.suptitle("Classical Methods Comparison", fontsize=14, fontweight='bold')
    col_titles = ["Original", "Color Threshold", "Otsu", "Ground Truth"]

    for j, title in enumerate(col_titles):
        axes[0, j].set_title(title, fontsize=12, fontweight='bold')

    for i, (img_path, mask_path) in enumerate(samples):
        img    = load_image(img_path)
        gt     = trimap_to_binary(load_mask(mask_path))
        color  = morphological_clean(color_threshold_segment(img))
        otsu   = morphological_clean(otsu_segment(img))

        # Display: original | color threshold mask | otsu mask | ground truth
        # Use cmap='gray' for masks
        # Turn off axes ticks for all subplots
        masks_to_show = [img, color, otsu, gt]
        for j, m in enumerate(masks_to_show):
            ax = axes[i, j]
            if j == 0:
                ax.imshow(m)
            else:
                ax.imshow(m, cmap='gray')
            ax.axis('off')
        # ────────────────────────────────────────────────────────────────────

    plt.tight_layout()
    plt.show()

compare_classical_methods(dataset, n=5)


---
## Part 3 — SAM ViT-B

We now move to the Segment Anything Model. SAM takes an image and a **prompt** as input and returns a precise mask.

Three prompt strategies, three exercises:
- **3A** — Point prompt: click on the object
- **3B** — Box prompt: draw a bounding box around the object
- **3C** — Automatic: no prompt at all — SAM segments everything


In [ ]:
# ── Load SAM ViT-B ───────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT = "sam_vit_b_01ec64.pth"

sam = sam_model_registry["vit_b"](checkpoint=CHECKPOINT)
sam.eval()
sam.to(DEVICE)

predictor = SamPredictor(sam)
print(f"✅ SAM ViT-B loaded on {DEVICE}")


### Exercise 3A — Point Prompt
**The idea:** you click one point on the pet. SAM returns three candidate masks ranked by confidence.  
You then pick the best one.


#### TODO 9 — Encode the image into SAM's embedding
Call `predictor.set_image()` to compute the image embedding.  
This is the expensive step — it runs the ViT-B encoder once per image.


In [ ]:
# Load a test image for Part 3
test_img_path, test_mask_path = dataset[0]
test_img  = load_image(test_img_path)
test_gt   = trimap_to_binary(load_mask(test_mask_path))

print(f"Image: {test_img_path.name}  shape: {test_img.shape}")

# Call predictor.set_image() with test_img
# Note: SAM expects a numpy array in RGB format — which is what load_image returns
predictor.set_image(test_img)
# ────────────────────────────────────────────────────────────────────────────

print("✅ Image encoded")


#### TODO 10 — Define a point prompt
Define a single foreground point at the **center of the image**.  
SAM expects points as numpy arrays of shape `(N, 2)` and labels as `(N,)`.  
Label `1` = foreground, `0` = background.


In [ ]:
# Define:
#   point_coords: numpy array shape (1, 2) with [x, y] at image center
#   point_labels: numpy array shape (1,) with value 1 (foreground)
# Hint: image center is at (width//2, height//2)
#       image shape is (H, W, C) so width = test_img.shape[1], height = test_img.shape[0]

h, w = test_img.shape[:2]
point_coords = np.array([[w // 2, h // 2]])
point_labels = np.array([1])
# ────────────────────────────────────────────────────────────────────────────

print(f"Point coords : {point_coords}")
print(f"Point labels : {point_labels}")


#### TODO 11 — Run SAM prediction with point prompt
Call `predictor.predict()` with your point coordinates and labels.  
Collect the three output masks and their IoU scores.


In [ ]:
# Call predictor.predict() with:
#   point_coords = point_coords
#   point_labels = point_labels
#   multimask_output = True  ← always True to get 3 masks
# The function returns: masks, scores, logits
# masks shape: (3, H, W) — three binary masks
# scores shape: (3,) — confidence score for each mask

masks, scores, logits = predictor.predict(
    point_coords=point_coords,
    point_labels=point_labels,
    multimask_output=True,
)
# ────────────────────────────────────────────────────────────────────────────

print(f"Masks shape  : {masks.shape}")
print(f"IoU scores   : {scores.round(3)}")


#### TODO 12 — Visualize the three SAM masks
Display the three masks SAM returned, each with its confidence score as title.  
Use a colored semi-transparent overlay on the original image.


In [ ]:
def show_mask(mask, ax, color=None, alpha=0.5):
    """Overlay a binary mask on an existing axes with a given color."""
    if color is None:
        color = np.array([30/255, 144/255, 255/255])
    h, w = mask.shape
    mask_img = np.zeros((h, w, 4))
    mask_img[mask == 1] = [*color, alpha]
    ax.imshow(mask_img)

def show_point(coords, labels, ax):
    """Draw prompt points on axes (green=fg, red=bg)."""
    for coord, label in zip(coords, labels):
        color = 'green' if label == 1 else 'red'
        ax.scatter(coord[0], coord[1], c=color, s=200, marker='*',
                   edgecolors='white', linewidths=1.5, zorder=5)

# Create a figure with 1 row and 3 columns
# For each of the 3 masks:
#   - Display test_img as background
#   - Call show_mask() to overlay the mask
#   - Call show_point() to show the prompt point
#   - Set title to f"Mask {i+1} — Score: {scores[i]:.3f}"
# Add a main title: "SAM Point Prompt — 3 Output Masks"

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('SAM Point Prompt — 3 Output Masks', fontsize=14, fontweight='bold')
for i in range(3):
    axes[i].imshow(test_img)
    show_mask(masks[i].astype(int), axes[i])
    show_point(point_coords, point_labels, axes[i])
    axes[i].set_title(f'Mask {i+1} — Score: {scores[i]:.3f}')
    axes[i].axis('off')
plt.tight_layout()
plt.show()
# ────────────────────────────────────────────────────────────────────────────


###  Reflection — Point Prompt
**Q: SAM returned three masks. Which one is best? Why did SAM return three instead of one?**

Le masque avec le score le plus élevé (généralement Mask 3) est le meilleur car il correspond le mieux à l'objet entier. SAM retourne trois masques car un seul point est ambigu : il peut appartenir à différents objets à différentes granularités (pixel, objet partiel, objet complet). Les trois masques représentent ces hypothèses à différentes échelles, permettant à l'utilisateur de choisir la plus adaptée.


---
### Exercise 3B — Box Prompt
**The idea:** instead of a point, we give SAM a bounding box around the pet.  
We compute this box automatically from the ground truth mask.


#### TODO 13 — Compute bounding box from ground truth mask
Find the tight bounding box around the foreground region of the binary mask.  
SAM expects boxes as `[x_min, y_min, x_max, y_max]`.


In [ ]:
def mask_to_bbox(binary_mask):
    """
    Compute bounding box of foreground region in a binary mask.
    Args:
        binary_mask: numpy array (H, W) with values 0 and 1
    Returns:
        numpy array [x_min, y_min, x_max, y_max]
    """
    # Use np.where to find rows and columns where mask == 1
    # x_min = cols.min(), x_max = cols.max()
    # y_min = rows.min(), y_max = rows.max()
    # Return as numpy array: np.array([x_min, y_min, x_max, y_max])
    rows, cols = np.where(binary_mask == 1)
    x_min, x_max = cols.min(), cols.max()
    y_min, y_max = rows.min(), rows.max()
    return np.array([x_min, y_min, x_max, y_max])
    # ────────────────────────────────────────────────────────────────────────

# Test
bbox = mask_to_bbox(test_gt)
print(f"Bounding box: {bbox}")


#### TODO 14 — Run SAM with box prompt
Pass the bounding box to `predictor.predict()` using the `box` argument.


In [ ]:
# Call predictor.predict() with:
#   box = bbox[None, :]   ← SAM expects shape (1, 4)
#   multimask_output = False  ← box prompts usually return 1 mask
# Collect masks, scores, logits

masks, scores, logits = predictor.predict(
    box=bbox[None, :],
    multimask_output=False,
)
# ────────────────────────────────────────────────────────────────────────────

box_mask = masks[0]
print(f"Box mask shape: {box_mask.shape}, score: {scores[0]:.3f}")


#### TODO 15 — Compare point vs box prompt vs ground truth
Display 3 images side by side: **point prompt result | box prompt result | ground truth**  
Do this for 3 different images from the dataset.


In [ ]:
def compare_prompts(dataset, predictor, n=3):
    """
    Compare point prompt vs box prompt vs ground truth for n images.
    """
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    fig.suptitle("SAM: Point Prompt vs Box Prompt vs Ground Truth",
                 fontsize=14, fontweight='bold')
    col_titles = ["Point Prompt", "Box Prompt", "Ground Truth"]
    for j, t in enumerate(col_titles):
        axes[0, j].set_title(t, fontsize=12, fontweight='bold')

    for i, (img_path, mask_path) in enumerate(dataset[:n]):
        img = load_image(img_path)
        gt  = trimap_to_binary(load_mask(mask_path))
        box = mask_to_bbox(gt)

        # 1. Call predictor.set_image(img)
        # 2. Get point prompt mask (center point, label 1, multimask_output=True, take mask[0])
        # 3. Get box prompt mask (box prompt, multimask_output=False, take mask[0])
        # 4. Display point mask | box mask | ground truth
        #    Use show_mask() helper for overlays
        predictor.set_image(img)
        h, w = img.shape[:2]
        pt_coords = np.array([[w // 2, h // 2]])
        pt_labels = np.array([1])
        pt_masks, _, _ = predictor.predict(point_coords=pt_coords, point_labels=pt_labels, multimask_output=True)
        pt_mask = pt_masks[0]

        bx_masks, _, _ = predictor.predict(box=box[None, :], multimask_output=False)
        bx_mask = bx_masks[0]

        for j, (m, title) in enumerate([(pt_mask, 'Point Prompt'), (bx_mask, 'Box Prompt'), (gt, 'Ground Truth')]):
            axes[i, j].imshow(img)
            show_mask(m.astype(int), axes[i, j])
            axes[i, j].axis('off')
            if i == 0:
                axes[i, j].set_title(title, fontsize=12, fontweight='bold')
        # ─────────────────────────────────────────────────────────────────────

    plt.tight_layout()
    plt.show()

compare_prompts(dataset, predictor, n=3)


---
### Exercise 3C — Automatic Mask Generation
**The idea:** no prompt at all. SAM runs on a grid of points and segments everything it finds.


In [ ]:
# ── Automatic mask generator — given, do not modify ────────────────────────
mask_generator = SamAutomaticMaskGenerator(
    model=sam,
    points_per_side=8,           # reduced for CPU speed
    pred_iou_thresh=0.88,
    stability_score_thresh=0.95,
    min_mask_region_area=500,
)
print(" Automatic mask generator ready")


#### TODO 16 — Run automatic mask generation
Run the mask generator on one image and collect all detected masks.


In [ ]:
auto_img_path, auto_mask_path = dataset[1]
auto_img = load_image(auto_img_path)

# Call mask_generator.generate(auto_img) and store the result in `auto_masks`
auto_masks = mask_generator.generate(auto_img)
# ────────────────────────────────────────────────────────────────────────────

print(f"✅ Found {len(auto_masks)} masks automatically")
print(f"Keys in each mask dict: {list(auto_masks[0].keys())}")


#### TODO 17 — Visualize all automatic masks
Write a function that draws each mask with a different random color overlaid on the image.


In [ ]:
def show_all_masks(image, masks, title="Automatic Masks"):
    """
    Display image with all automatic masks overlaid, each with a different color.
    Args:
        image: RGB numpy array
        masks: list of mask dicts from SamAutomaticMaskGenerator
        title: plot title
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    # Left: original image
    axes[0].imshow(image)
    axes[0].set_title("Original Image")
    axes[0].axis('off')

    # Right: image with all masks overlaid
    axes[1].imshow(image)
    axes[1].set_title(f"{len(masks)} Masks Found")

    # For each mask in masks:
    #   1. Generate a random RGB color: np.random.rand(3)
    #   2. Call show_mask(mask['segmentation'], axes[1], color=color, alpha=0.4)
    # Turn off axes[1] ticks
    for m in masks:
        color = np.random.rand(3)
        show_mask(m['segmentation'].astype(int), axes[1], color=color, alpha=0.4)
    axes[1].axis('off')
    # ────────────────────────────────────────────────────────────────────────

    plt.tight_layout()
    plt.show()

show_all_masks(auto_img, auto_masks, f"Automatic Segmentation — {auto_img_path.name}")


#### TODO 18 — Sort masks by confidence and display top 3
Sort all automatic masks by their `predicted_iou` score (descending).  
Display the top 3 individually with their score and area.


In [ ]:
# 1. Print total number of masks found
# 2. Sort auto_masks by 'predicted_iou' descending
# 3. Display top 3 masks individually (3 columns)
#    Each subplot: original image + one mask overlay
#    Title: f"Rank {i+1} | IoU: {m['predicted_iou']:.3f} | Area: {m['area']} px"

print(f'Total masks found: {len(auto_masks)}')
sorted_masks = sorted(auto_masks, key=lambda x: x['predicted_iou'], reverse=True)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Top 3 Automatic Masks by Confidence', fontsize=14, fontweight='bold')
for i, m in enumerate(sorted_masks[:3]):
    axes[i].imshow(auto_img)
    show_mask(m['segmentation'].astype(int), axes[i])
    axes[i].set_title(f"Rank {i+1} | IoU: {m['predicted_iou']:.3f} | Area: {m['area']} px")
    axes[i].axis('off')
plt.tight_layout()
plt.show()
# ────────────────────────────────────────────────────────────────────────────


### ✍️ Reflection — Automatic Generation
**Q: SAM found N masks automatically. Are all of them meaningful? What determines whether a region gets its own mask?**

Non, tous les masques ne sont pas nécessairement significatifs : SAM peut détecter des régions texturées ou des zones de fond comme des objets distincts. Une région obtient son propre masque si elle présente une cohérence visuelle suffisante (bords nets, homogénéité de texture/couleur) autour des points de grille. Les seuils `pred_iou_thresh`, `stability_score_thresh` et `min_mask_region_area` filtrent les masques de faible qualité.


---
## Part 4 — Metrics: Measuring What We See

You have seen IoU and DICE in the lecture. Now implement them from scratch  
and use them to rigorously compare all three segmentation approaches.

Recall:
$$\text{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

$$\text{DICE}(A, B) = \frac{2|A \cap B|}{|A| + |B|}$$


### TODO 19 — Implement IoU from scratch


In [ ]:
def compute_iou(pred_mask, gt_mask):
    """
    Compute Intersection over Union between two binary masks.
    Args:
        pred_mask: numpy array (H, W) with values 0 and 1
        gt_mask  : numpy array (H, W) with values 0 and 1
    Returns:
        float in [0, 1]
    """

    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    if union == 0:
        return 1.0
    return float(intersection) / float(union)
    # ────────────────────────────────────────────────────────────────────────

#  checks
perfect = np.array([[1, 1, 0], [0, 1, 0]])
assert compute_iou(perfect, perfect) == 1.0, "Perfect overlap should give IoU=1"
empty   = np.zeros_like(perfect)
assert compute_iou(perfect, empty)   == 0.0, "No overlap should give IoU=0"
print("✅ IoU sanity checks passed")


### TODO 20 — Implement DICE from scratch


In [ ]:
def compute_dice(pred_mask, gt_mask):
    """
    Compute DICE score between two binary masks.
    Args:
        pred_mask: numpy array (H, W) with values 0 and 1
        gt_mask  : numpy array (H, W) with values 0 and 1
    Returns:
        float in [0, 1]
    """

    intersection = np.logical_and(pred_mask, gt_mask).sum()
    denom = pred_mask.sum() + gt_mask.sum()
    if denom == 0:
        return 1.0
    return float(2 * intersection) / float(denom)
    # ────────────────────────────────────────────────────────────────────────

#  checks
assert compute_dice(perfect, perfect) == 1.0, "Perfect overlap should give DICE=1"
assert compute_dice(perfect, empty)   == 0.0, "No overlap should give DICE=0"
print("✅ DICE sanity checks passed")


### TODO 21 — Evaluate all three methods on 50 images
Compute mean IoU and mean DICE for:
- Color thresholding
- SAM point prompt (center point)
- SAM box prompt (from ground truth bbox)


In [ ]:
def evaluate_method(dataset, predictor, method="color", n=50):
    """
    Evaluate a segmentation method on n images.
    Returns dict with 'iou' and 'dice' lists.
    """
    ious, dices = [], []
    print(f"Evaluating: {method} on {n} images ...")

    for img_path, mask_path in dataset[:n]:
        img = load_image(img_path)
        gt  = trimap_to_binary(load_mask(mask_path))

        if method == "color":
            pred = morphological_clean(color_threshold_segment(img))

        elif method == "otsu":
            pred = morphological_clean(otsu_segment(img))

        elif method == "sam_point":

            predictor.set_image(img)
            h, w = img.shape[:2]
            pt_coords = np.array([[w // 2, h // 2]])
            pt_labels = np.array([1])
            ms, _, _ = predictor.predict(point_coords=pt_coords, point_labels=pt_labels, multimask_output=True)
            pred = ms[0].astype(np.uint8)
            # ────────────────────────────────────────────────────────────────

        elif method == "sam_box":
            # ── TODO 21b ────────────────────────────────────────────────────

            predictor.set_image(img)
            box = mask_to_bbox(gt)
            ms, _, _ = predictor.predict(box=box[None, :], multimask_output=False)
            pred = ms[0].astype(np.uint8)
            # ────────────────────────────────────────────────────────────────

        # Resize pred to gt size if needed
        if pred.shape != gt.shape:
            pred = cv2.resize(pred.astype(np.uint8), (gt.shape[1], gt.shape[0]))

        ious.append(compute_iou(pred.astype(bool), gt.astype(bool)))
        dices.append(compute_dice(pred.astype(bool), gt.astype(bool)))

    return {"iou": ious, "dice": dices}

# ── TODO 21c ─────────────────────────────────────────────────────────────────
# Run evaluate_method for all 4 methods and store results in a dict:
# results = {
#     "Color Threshold" : evaluate_method(dataset, predictor, method="color"),
#     "Otsu"            : evaluate_method(dataset, predictor, method="otsu"),
#     "SAM Point"       : evaluate_method(dataset, predictor, method="sam_point"),
#     "SAM Box"         : evaluate_method(dataset, predictor, method="sam_box"),
# }
results = {
    'Color Threshold': evaluate_method(dataset, predictor, method='color'),
    'Otsu'           : evaluate_method(dataset, predictor, method='otsu'),
    'SAM Point'      : evaluate_method(dataset, predictor, method='sam_point'),
    'SAM Box'        : evaluate_method(dataset, predictor, method='sam_box'),
}
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
# ── Print results table (given — run after TODO 21) ─────────────────────────
print("\n" + "="*55)
print(f"{'Method':<20} {'Mean IoU':>10} {'Mean DICE':>10}")
print("="*55)
for method_name, res in results.items():
    print(f"{method_name:<20} {np.mean(res['iou']):>10.3f} {np.mean(res['dice']):>10.3f}")
print("="*55)


### TODO 22 — Plot results
Two plots:
1. **Bar chart** comparing mean IoU across all methods
2. **Scatter plot** of per-image IoU: SAM point vs SAM box (one dot per image)


In [ ]:
def plot_results(results):
    """
    Plot 1: bar chart of mean IoU per method
    Plot 2: scatter plot SAM point vs SAM box per image
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Segmentation Evaluation Results", fontsize=14, fontweight='bold')


    method_names = list(results.keys())
    mean_ious = [np.mean(results[m]['iou']) for m in method_names]
    axes[0].bar(method_names, mean_ious, color=['#e07b54', '#e0b554', '#54a0e0', '#54e07b'])
    axes[0].set_ylim(0, 1)
    axes[0].set_ylabel('Mean IoU')
    axes[0].set_title('Mean IoU per Method')
    for k, v in enumerate(mean_ious):
        axes[0].text(k, v + 0.01, f'{v:.3f}', ha='center')
    # ────────────────────────────────────────────────────────────────────────

    # ── TODO 22b — Scatter plot ──────────────────────────────────────────────

    pt_ious  = results['SAM Point']['iou']
    box_ious = results['SAM Box']['iou']
    axes[1].scatter(pt_ious, box_ious, alpha=0.7, edgecolors='k', linewidths=0.5)
    axes[1].plot([0, 1], [0, 1], 'r--', label='y=x')
    axes[1].set_xlabel('SAM Point IoU')
    axes[1].set_ylabel('SAM Box IoU')
    axes[1].set_title('Per-image IoU: Point vs Box')
    axes[1].legend()
    # ────────────────────────────────────────────────────────────────────────

    plt.tight_layout()
    plt.show()

plot_results(results)


### TODO 23 — Failure analysis
Find the 3 images where SAM box prompt got the **lowest IoU**.  
Display each with: original image | SAM box mask | ground truth | IoU score as title.  
Write 3 sentences below explaining what went wrong visually.


In [ ]:
def failure_analysis(dataset, predictor, results, method_key="SAM Box", n_failures=3):
    """
    Find n worst predictions and display them.
    """

    box_ious = results[method_key]['iou']
    sorted_indices = np.argsort(box_ious)[:n_failures]

    fig, axes = plt.subplots(n_failures, 3, figsize=(12, 4 * n_failures))
    fig.suptitle(f'Failure Analysis — {method_key} (Lowest IoU)', fontsize=14, fontweight='bold')

    for row, idx in enumerate(sorted_indices):
        img_path, mask_path = dataset[idx]
        img = load_image(img_path)
        gt  = trimap_to_binary(load_mask(mask_path))
        box = mask_to_bbox(gt)
        predictor.set_image(img)
        ms, _, _ = predictor.predict(box=box[None, :], multimask_output=False)
        pred = ms[0].astype(int)
        iou_val = box_ious[idx]

        axes[row, 0].imshow(img)
        axes[row, 0].set_title(f'Original (IoU={iou_val:.3f})')
        axes[row, 0].axis('off')

        axes[row, 1].imshow(img)
        show_mask(pred, axes[row, 1])
        axes[row, 1].set_title('SAM Box Mask')
        axes[row, 1].axis('off')

        axes[row, 2].imshow(gt, cmap='gray')
        axes[row, 2].set_title('Ground Truth')
        axes[row, 2].axis('off')

    plt.tight_layout()
    plt.show()
    # ────────────────────────────────────────────────────────────────────────

failure_analysis(dataset, predictor, results)


### ✍️ Failure Analysis — Written Response
**Look at the 3 failure cases above. Write 3 sentences explaining what went wrong and why.**

1. Dans certains cas, le bounding box ground truth englobe une grande portion de fond (herbe, canapé) de couleur similaire au pelage de l'animal, ce qui pousse SAM à inclure ces régions dans le masque prédit.
2. Pour les animaux à poil long ou aux contours flous, SAM tend à sous-segmenter certaines zones périphériques car les bords dans la boîte sont peu marqués.
3. Les images avec plusieurs animaux ou des reflets prononcés peuvent confondre SAM : la boîte contient plusieurs objets saillants et le modèle ne sait pas lequel privilégier.
